In [1]:
# ! pip install pypower --quiet

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import os

SEED = 2024
torch.manual_seed(SEED)

# The solution of the AC-OPF provides a setpoint for power generations in order to satisfy a specific state of electricity consumtion !!!!!!!


In [3]:
# Load the network for Case14
net = {
    "baseMVA": 100.0,
## area data
    "areas": np.array([[1, 4]]),
## bus data
###	bus_i	type	Pd	Qd	Gs	Bs	area	Vm	Va	baseKV	zone	Vmax	Vmin
    "bus": np.array([
                    [1,	 3,	 0.0,	 0.0,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[2,	 2,	 21.7,	 12.7,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,    0.94000],
                	[3,	 2,	 94.2,	 19.0,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,    0.94000],
                	[4,	 1,	 47.8,	 -3.9,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[5,  1,	 7.6,	 1.6,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[6,	 2,	 11.2,	 7.5,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[7,	 1,	 0.0,	 0.0,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[8,	 2,	 0.0,	 0.0,	 0.0,	 0.0,	 1,     1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[9,	 1,	 29.5,	 16.6,	 0.0,	 19.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,    0.94000],
                	[10, 1,	 9.0,	 5.8,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[11, 1,	 3.5,	 1.8,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[12, 1,	 6.1,	 1.6,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[13, 1,	 13.5,	 5.8,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[14, 1,	 14.9,	 5.0,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                ]),
    
## generator data
###	bus	Pg	Qg	Qmax	Qmin	Vg	mBase	status	Pmax	Pmin
    "gen": np.array([
                	[1,	 170.0,	 5.0,	 10.0,	  0.0,	 1.0,	 100.0,	 1,	 340,  0.0],
                	[2,	 29.5,	 0.0,	 30.0,	 -30.0,	 1.0,	 100.0,	 1,	 59,   0.0],
                	[3,	 0.0,	 20.0,	 40.0,	  0.0,	 1.0,	 100.0,	 1,	 0,	   0.0],
                	[6,	 0.0,	 9.0,	 24.0,	 -6.0,	 1.0,	 100.0,	 1,	 0,	   0.0],
                	[8,	 0.0,	 9.0,	 24.0,	 -6.0,	 1.0,	 100.0,	 1,	 0,	   0.0]
                 ]),
    
## generator cost data
###	2	startup	shutdown	n	c(n-1)	...	c0
    "gencost": np.array([
                	[2,	 0.0,	 0.0,	 3,	   0.000000,	   7.920951,	   0.000000],
                	[2,	 0.0,	 0.0,	 3,	   0.000000,	  23.269494,	   0.000000],
                	[2,	 0.0,	 0.0,	 3,	   0.000000,	   0.000000,	   0.000000],
                	[2,	 0.0,	 0.0,	 3,	   0.000000,	   0.000000,	   0.000000],
                	[2,	 0.0,	 0.0,	 3,	   0.000000,	   0.000000,	   0.000000]
                ]),
    
## branch data
###	fbus	tbus	r	x	b	rateA	rateB	rateC	ratio	angle	status	angmin	angmax
    "branch": np.array([
                    [1,	 2,	 0.01938,	 0.05917,	 0.0528, 472,	 472,	 472,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[1,	 5,	 0.05403,	 0.22304,	 0.0492, 128,	 128,	 128,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[2,	 3,	 0.04699,	 0.19797,	 0.0438, 145,	 145,	 145,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                    [2,	 4,	 0.05811,	 0.17632,	 0.034,	 158,	 158,	 158,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[2,	 5,	 0.05695,	 0.17388,	 0.0346, 161,	 161,	 161,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[3,	 4,	 0.06701,	 0.17103,	 0.0128, 160,	 160,	 160,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[4,	 5,	 0.01335,	 0.04211,	 0.0,	 664,	 664,	 664,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[4,	 7,	 0.0,	     0.20912,	 0.0,	 141,	 141,	 141,	 0.978,	 0.0,	 1,	 -30.0,	 30.0],
                	[4,	 9,	 0.0,	     0.55618,	 0.0,	 53,	 53,	 53,	 0.969,	 0.0,	 1,	 -30.0,	 30.0],
                	[5,  6,	 0.0,	     0.25202,	 0.0,	 117,	 117,	 117,	 0.932,	 0.0,	 1,	 -30.0,	 30.0],
                	[6,	 11, 0.09498,	 0.1989,     0.0,	 134,	 134,	 134,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
               	    [6,	 12, 0.12291,	 0.25581,	 0.0,	 104,	 104,	 104,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[6,	 13, 0.06615,	 0.13027,	 0.0,	 201,	 201,	 201,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[7,	 8,	 0.0,	     0.17615,	 0.0,	 167,	 167,	 167,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[7,	 9,	 0.0,	     0.11001,	 0.0,	 267,	 267,	 267,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[9,	 10, 0.03181,	 0.0845,     0.0,	 325,	 325,	 325,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[9,	 14, 0.12711,	 0.27038,	 0.0,	 99,	 99,	 99,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[10, 11, 0.08205,	 0.19207,	 0.0,	 141,	 141,	 141,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[12, 13, 0.22092,	 0.19988,	 0.0,	 99,	 99,	 99,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[13, 14, 0.17093,	 0.34802,	 0.0,	 76,	 76,	 76,     0.0,	 0.0,	 1,	 -30.0,	 30.0]
                ])
}

In [4]:
bus = pd.DataFrame(net['bus'],
                   columns=['bus_i', 'type',	'Pd',	'Qd',	'Gs',	'Bs',
                            'area',	'Vm',	'Va',	'baseKV',	'zone',	'Vmax',	'Vmin'])
print(f"n_bus:{len(bus)}")
bus

n_bus:14


,bus_i,type,Pd,Qd,Gs,Bs,area,Vm,Va,baseKV,zone,Vmax,Vmin
0,1.0,3.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.06,0.94
1,2.0,2.0,21.7,12.7,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.06,0.94
2,3.0,2.0,94.2,19.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.06,0.94
3,4.0,1.0,47.8,-3.9,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.06,0.94
4,5.0,1.0,7.6,1.6,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.06,0.94
5,6.0,2.0,11.2,7.5,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.06,0.94
6,7.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.06,0.94
7,8.0,2.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.06,0.94
8,9.0,1.0,29.5,16.6,0.0,19.0,1.0,1.0,0.0,1.0,1.0,1.06,0.94
9,10.0,1.0,9.0,5.8,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.06,0.94


In [5]:
gen = pd.DataFrame(net['gen'],
                   columns = ['bus',	'Pg',	'Qg',
                              'Qmax',	'Qmin',	'Vg',
                              'mBase',	'status',	'Pmax',	'Pmin'])
gen

,bus,Pg,Qg,Qmax,Qmin,Vg,mBase,status,Pmax,Pmin
0,1.0,170.0,5.0,10.0,0.0,1.0,100.0,1.0,340.0,0.0
1,2.0,29.5,0.0,30.0,-30.0,1.0,100.0,1.0,59.0,0.0
2,3.0,0.0,20.0,40.0,0.0,1.0,100.0,1.0,0.0,0.0
3,6.0,0.0,9.0,24.0,-6.0,1.0,100.0,1.0,0.0,0.0
4,8.0,0.0,9.0,24.0,-6.0,1.0,100.0,1.0,0.0,0.0


In [6]:
# The bus variables indicate where the generator is connected on the grid !!!!!!

In [7]:
line = pd.DataFrame(net['branch'],
                    columns = ['fbus',	'tbus',	'r',	'x',	'b',
                               'rateA',	'rateB',	'rateC',	'ratio',
                               'angle',	'status',	'angmin',	'angmax'] )
line

,fbus,tbus,r,x,b,rateA,rateB,rateC,ratio,angle,status,angmin,angmax
0,1.0,2.0,0.01938,0.05917,0.0528,472.0,472.0,472.0,0.000,0.0,1.0,-30.0,30.0
1,1.0,5.0,0.05403,0.22304,0.0492,128.0,128.0,128.0,0.000,0.0,1.0,-30.0,30.0
2,2.0,3.0,0.04699,0.19797,0.0438,145.0,145.0,145.0,0.000,0.0,1.0,-30.0,30.0
3,2.0,4.0,0.05811,0.17632,0.0340,158.0,158.0,158.0,0.000,0.0,1.0,-30.0,30.0
4,2.0,5.0,0.05695,0.17388,0.0346,161.0,161.0,161.0,0.000,0.0,1.0,-30.0,30.0
5,3.0,4.0,0.06701,0.17103,0.0128,160.0,160.0,160.0,0.000,0.0,1.0,-30.0,30.0
6,4.0,5.0,0.01335,0.04211,0.0000,664.0,664.0,664.0,0.000,0.0,1.0,-30.0,30.0
7,4.0,7.0,0.00000,0.20912,0.0000,141.0,141.0,141.0,0.978,0.0,1.0,-30.0,30.0
8,4.0,9.0,0.00000,0.55618,0.0000,53.0,53.0,53.0,0.969,0.0,1.0,-30.0,30.0
9,5.0,6.0,0.00000,0.25202,0.0000,117.0,117.0,117.0,0.932,0.0,1.0,-30.0,30.0


In [8]:
# lines connecting from_bus to to_bus with some specific parameters, e.g. resistance (r) and reactance (x).

In [9]:
gen_cost = pd.DataFrame(net['gencost'],
                        columns = ['order_poly', 'startup', 'shutdown', 'num_coefficient', 'c2', 'c1', 'c0'])
gen_cost

,order_poly,startup,shutdown,num_coefficient,c2,c1,c0
0,2.0,0.0,0.0,3.0,0.0,7.920951,0.0
1,2.0,0.0,0.0,3.0,0.0,23.269494,0.0
2,2.0,0.0,0.0,3.0,0.0,0.000000,0.0
3,2.0,0.0,0.0,3.0,0.0,0.000000,0.0
4,2.0,0.0,0.0,3.0,0.0,0.000000,0.0


In [10]:
# First, we use PyPower to solve the complex AC-OPF problem

In [11]:
#! pip install pyrlu


# Machine learning for AC-OPF

In [12]:
"""================================Training Data Generation================================"""

'================================Training Data Generation================================'

In [13]:
from pypower import idx_bus, idx_gen, idx_brch
from tqdm import tqdm

# Data exploration

In [14]:
df = pd.read_csv('./data/pglib_opf_case5_pjm.csv')
df.head()

,load1:pl,load2:pl,load3:pl,load1:ql,load2:ql,load3:ql,gen1:pg,gen2:pg,gen3:pg,gen4:pg,...,line3:p_fr_max,line4:p_fr_max,line5:p_fr_max,line6:p_fr_max,line1:q_fr_max,line2:q_fr_max,line3:q_fr_max,line4:q_fr_max,line5:q_fr_max,line6:q_fr_max
0,6.570765,2.275993,4.051883,4.376283,0.628149,4.026378,0.4,1.7,4.876424,1.558644,...,-0.000002,-0.000003,-2.720498e-07,-0.000006,0.000000e+00,0.0,-7.933284e-07,-1.873404e-06,0.0,-3.069352e-06
1,6.552550,1.704638,4.090173,4.400894,0.724693,3.937011,0.4,1.7,4.317437,1.531758,...,-0.000002,-0.000003,-2.624331e-07,-0.000006,0.000000e+00,0.0,-7.848656e-07,-1.892890e-06,0.0,-2.936123e-06
2,5.645406,2.029723,5.955067,1.377338,0.865517,3.240306,0.4,1.7,5.199999,1.490930,...,-0.000002,-0.000002,0.000000e+00,-0.000280,-1.365476e-07,0.0,0.000000e+00,-8.479213e-07,0.0,-3.177477e-07
3,4.778645,2.417017,6.361179,0.843277,0.881531,2.635903,0.4,1.7,5.200000,1.644658,...,-0.000001,-0.000001,0.000000e+00,-0.647117,-3.141235e-07,0.0,0.000000e+00,-7.238938e-07,0.0,-6.550166e-09
4,5.690040,2.458801,5.745494,2.003454,1.119158,3.286294,0.4,1.7,5.200000,1.684051,...,-0.000002,-0.000002,-5.028490e-09,-0.000067,0.000000e+00,0.0,0.000000e+00,-9.201327e-07,0.0,-6.686726e-07


In [15]:
df.shape

(10000, 104)

In [16]:
df.columns

Index(['load1:pl', 'load2:pl', 'load3:pl', 'load1:ql', 'load2:ql', 'load3:ql',
       'gen1:pg', 'gen2:pg', 'gen3:pg', 'gen4:pg',
       ...
       'line3:p_fr_max', 'line4:p_fr_max', 'line5:p_fr_max', 'line6:p_fr_max',
       'line1:q_fr_max', 'line2:q_fr_max', 'line3:q_fr_max', 'line4:q_fr_max',
       'line5:q_fr_max', 'line6:q_fr_max'],
      dtype='object', length=104)

In [17]:
""" We can see that this is dataset of 10000 rows by 104 columns.
    Each row represents the feature and solution of the AC-OPF.
    Let's define the inputs and outputs for the machine learning based on the AC-OPF formulation we have learned.
"""

" We can see that this is dataset of 10000 rows by 104 columns.\n    Each row represents the feature and solution of the AC-OPF.\n    Let's define the inputs and outputs for the machine learning based on the AC-OPF formulation we have learned.\n"

In [18]:
""" For example, columns 0:3 represents the active power consumption for the loads, while comlumns 3:6 represent the reactive power consumption of the loads.
    We can now separete them into a variable called "input", which means they are the input parameters for the AC-OPF problem.
"""

' For example, columns 0:3 represents the active power consumption for the loads, while comlumns 3:6 represent the reactive power consumption of the loads.\n    We can now separete them into a variable called "input", which means they are the input parameters for the AC-OPF problem.\n'

In [19]:
# define input columns

load_p = list(df.columns[0:3])
load_q = list(df.columns[3:6])
inputs = load_p + load_q
print(inputs)

['load1:pl', 'load2:pl', 'load3:pl', 'load1:ql', 'load2:ql', 'load3:ql']


In [20]:
""" We can do the same with the outputs, the optimal solution for the AC-OPF problem given load demand,
    which includes the active and reactive power at each generator as well as the voltage magnitude and angle at each bus.
"""

' We can do the same with the outputs, the optimal solution for the AC-OPF problem given load demand,\n    which includes the active and reactive power at each generator as well as the voltage magnitude and angle at each bus.\n'

In [21]:
# Define output columns

gen_p = list(df.columns[6:11])
gen_q = list(df.columns[11:16])

In [22]:
gen_p

['gen1:pg', 'gen2:pg', 'gen3:pg', 'gen4:pg', 'gen5:pg']

In [23]:
def convert(x):
    #Remove spaces around the '+' or '-' before 'j'
    x = x.replace(" + ", "+").replace(" - ", "-").replace(" j ", "j").strip()
    return np.complex64(x)

In [24]:
df.columns[21:26]

Index(['bus1:v_bus', 'bus2:v_bus', 'bus3:v_bus', 'bus4:v_bus', 'bus5:v_bus'], dtype='object')

In [25]:
bus_vm = []
bus_va = []
for bus_v_column in df.columns[21:26]:
    df[bus_v_column + '_mag'] = df[bus_v_column].apply(convert).apply(np.abs)
    bus_vm.append(bus_v_column + '_mag')
    
    df[bus_v_column + '_ang'] = df[bus_v_column].apply(convert).apply(np.angle)
    ## the follow line of code is to correct a bug in the OPFLearn dataset
    df[bus_v_column + '_ang'] = -1*np.rad2deg(df[bus_v_column + '_ang'].values)
    bus_va.append(bus_v_column + '_ang')
    
outputs = gen_p + gen_q + bus_vm + bus_va
print(outputs)
    

['gen1:pg', 'gen2:pg', 'gen3:pg', 'gen4:pg', 'gen5:pg', 'gen1:qg', 'gen2:qg', 'gen3:qg', 'gen4:qg', 'gen5:qg', 'bus1:v_bus_mag', 'bus2:v_bus_mag', 'bus3:v_bus_mag', 'bus4:v_bus_mag', 'bus5:v_bus_mag', 'bus1:v_bus_ang', 'bus2:v_bus_ang', 'bus3:v_bus_ang', 'bus4:v_bus_ang', 'bus5:v_bus_ang']


In [26]:
df

,load1:pl,load2:pl,load3:pl,load1:ql,load2:ql,load3:ql,gen1:pg,gen2:pg,gen3:pg,gen4:pg,...,bus1:v_bus_mag,bus1:v_bus_ang,bus2:v_bus_mag,bus2:v_bus_ang,bus3:v_bus_mag,bus3:v_bus_ang,bus4:v_bus_mag,bus4:v_bus_ang,bus5:v_bus_mag,bus5:v_bus_ang
0,6.570765,2.275993,4.051883,4.376283,0.628149,4.026378,0.4,1.7,4.876424,1.558644e+00,...,1.088747,0.031466,1.027343,-0.047701,1.058290,-0.018030,1.055571,-1.044813e-31,1.100000,0.044278
1,6.552550,1.704638,4.090173,4.400894,0.724693,3.937011,0.4,1.7,4.317437,1.531758e+00,...,1.088817,0.032032,1.027087,-0.046907,1.058152,-0.017335,1.056337,-7.842915e-32,1.100000,0.044911
2,5.645406,2.029723,5.955067,1.377338,0.865517,3.240306,0.4,1.7,5.199999,1.490930e+00,...,1.081929,0.046786,1.080142,-0.020623,1.100000,0.003993,1.064614,-4.099984e-33,1.076183,0.061261
3,4.778645,2.417017,6.361179,0.843277,0.881531,2.635903,0.4,1.7,5.200000,1.644658e+00,...,1.073330,0.049556,1.082470,-0.011662,1.100000,0.007987,1.059389,3.173137e-36,1.064221,0.063242
4,5.690040,2.458801,5.745494,2.003454,1.119158,3.286294,0.4,1.7,5.200000,1.684051e+00,...,1.095219,0.043703,1.079232,-0.025925,1.100000,-0.002590,1.075028,2.562954e-33,1.093446,0.058019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,4.254369,4.712440,4.836094,2.668600,3.629953,1.640128,0.4,1.7,5.200000,1.622811e+00,...,1.093770,0.042283,1.039607,-0.028559,1.051908,-0.017452,1.076681,-8.360963e-35,1.100000,0.055911
9996,3.210700,1.671734,4.661965,1.612852,0.490558,2.501960,0.4,1.7,3.086005,1.944606e-07,...,1.075134,0.050848,1.078772,-0.002335,1.100000,0.005464,1.062197,3.296830e-36,1.066963,0.062918
9997,4.979303,3.477785,4.071830,2.229018,3.314802,2.390931,0.4,1.7,5.200000,5.078266e-01,...,1.094907,0.041556,1.049922,-0.026832,1.061713,-0.007820,1.073106,4.240605e-34,1.100000,0.054636
9998,2.047435,5.546288,4.167393,0.539519,4.802855,0.740298,0.4,1.7,5.200000,7.091980e-02,...,1.100000,0.048322,1.074188,-0.002144,1.072140,-0.003328,1.092085,2.087602e-34,1.097649,0.059401


In [27]:
""" Take a look into each of those columns.
    The following function plots the histogram of each variable, so it takes the whole 10000 samples for each variable and plots the distribution.
    On the x-axis we see the variable value and on the y-axis the frequency of samples that fit into a specific bucket or bin.
    We will be using 10 bins just to get an idea of the shape of the distribution.
    Notice that we are also annotating the mean of all the samples in black.
"""

' Take a look into each of those columns.\n    The following function plots the histogram of each variable, so it takes the whole 10000 samples for each variable and plots the distribution.\n    On the x-axis we see the variable value and on the y-axis the frequency of samples that fit into a specific bucket or bin.\n    We will be using 10 bins just to get an idea of the shape of the distribution.\n    Notice that we are also annotating the mean of all the samples in black.\n'

In [28]:
""" We can see that generators'setpoints are set always at the same level most of the time.
    Notice that some generators are generating "negative" reactive power.
    This mean, by convention, that sometimes these generators are consuming reactive power instead of generating.
    In the case of voltage magnitude, it is very constant around the mean with a few samples below that level.
    This is expected as the voltage magnitude values are constrained by the AC-OPF problem.
"""

' We can see that generators\'setpoints are set always at the same level most of the time.\n    Notice that some generators are generating "negative" reactive power.\n    This mean, by convention, that sometimes these generators are consuming reactive power instead of generating.\n    In the case of voltage magnitude, it is very constant around the mean with a few samples below that level.\n    This is expected as the voltage magnitude values are constrained by the AC-OPF problem.\n'

# Case Study: Traditional Machine learning for Solving AC-OPF

In [29]:
torch.cuda.is_available()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"=======Using {device} device==========")

=======Using cuda device==========


In [30]:
class OPFDataset(Dataset):
    def __init__(self, data, inputs, outputs):
        self.data = data
        self.inputs = inputs
        self.outputs = outputs
    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        input = torch.tensor(self.data.iloc[idx,:len(self.inputs)].values).to(torch.float32)
        output = torch.tensor(self.data.iloc[idx, len(self.inputs):].values).to(torch.float32)

        return input, output

In [31]:
data = df[inputs + outputs]
train_data = data.sample(frac=0.8, random_state=SEED)
test_data = data.drop(train_data.index)

In [32]:
train_dataloader = DataLoader(OPFDataset(train_data, inputs, outputs),
                             batch_size=32, shuffle=True)
test_dataloader = DataLoader(OPFDataset(test_data, inputs, outputs),
                            batch_size=32, shuffle=False)

# Evaluation Metric for AC-OPF

In [33]:
""" We have a trained model to "solve" AC-OPF, which predicts the power generation and bus voltage values given the electricity load demand.
    We use a mean squared error (MSE) loss function to train the NN model and evaluate it on the test dataset.
    ----------------------------------Butttttt, HOW DO WE KNOW THE NN IS GOOD ENOUGH TO SOLVE AC-OPF ?-------------------------------------
    General, we care about three metrics:
    1. Feasibility: The solution needs to satisfy the equality & inequality constraints of the optimization problem.
    2. Optimaly: The feasibile solution incurs the lowest cost compared with all other feasible solutions.
    3. Run-time complexity: We solve the problem fast.
"""

' We have a trained model to "solve" AC-OPF, which predicts the power generation and bus voltage values given the electricity load demand.\n    We use a mean squared error (MSE) loss function to train the NN model and evaluate it on the test dataset.\n    ----------------------------------Butttttt, HOW DO WE KNOW THE NN IS GOOD ENOUGH TO SOLVE AC-OPF ?-------------------------------------\n    General, we care about three metrics:\n    1. Feasibility: The solution needs to satisfy the equality & inequality constraints of the optimization problem.\n    2. Optimaly: The feasibile solution incurs the lowest cost compared with all other feasible solutions.\n    3. Run-time complexity: We solve the problem fast.\n'

In [34]:
batch_snapshot = next(iter(test_dataloader))
snapshot_x, snapshot_y = batch_snapshot[0][[0]], batch_snapshot[1][[0]]

#  Solution quality

In [35]:
 """ ==============================Prepare dataset=========================== """

if not os.path.exists('./data/pglib_opf_case5_pjm.csv'):
    ! mkdir data
    ! wget https://data.nrel.gov/system/files/177/pglib_opf_case5_pjm.csv -P data/

df = pd.read_csv('./data/pglib_opf_case5_pjm.csv')

df

,load1:pl,load2:pl,load3:pl,load1:ql,load2:ql,load3:ql,gen1:pg,gen2:pg,gen3:pg,gen4:pg,...,line3:p_fr_max,line4:p_fr_max,line5:p_fr_max,line6:p_fr_max,line1:q_fr_max,line2:q_fr_max,line3:q_fr_max,line4:q_fr_max,line5:q_fr_max,line6:q_fr_max
0,6.570765,2.275993,4.051883,4.376283,0.628149,4.026378,0.4,1.7,4.876424,1.558644e+00,...,-0.000002,-3.158833e-06,-2.720498e-07,-0.000006,0.000000e+00,0.0,-7.933284e-07,-1.873404e-06,0.000000e+00,-3.069352e-06
1,6.552550,1.704638,4.090173,4.400894,0.724693,3.937011,0.4,1.7,4.317437,1.531758e+00,...,-0.000002,-3.122458e-06,-2.624331e-07,-0.000006,0.000000e+00,0.0,-7.848656e-07,-1.892890e-06,0.000000e+00,-2.936123e-06
2,5.645406,2.029723,5.955067,1.377338,0.865517,3.240306,0.4,1.7,5.199999,1.490930e+00,...,-0.000002,-2.135219e-06,0.000000e+00,-0.000280,-1.365476e-07,0.0,0.000000e+00,-8.479213e-07,0.000000e+00,-3.177477e-07
3,4.778645,2.417017,6.361179,0.843277,0.881531,2.635903,0.4,1.7,5.200000,1.644658e+00,...,-0.000001,-1.325614e-06,0.000000e+00,-0.647117,-3.141235e-07,0.0,0.000000e+00,-7.238938e-07,0.000000e+00,-6.550166e-09
4,5.690040,2.458801,5.745494,2.003454,1.119158,3.286294,0.4,1.7,5.200000,1.684051e+00,...,-0.000002,-1.892448e-06,-5.028490e-09,-0.000067,0.000000e+00,0.0,0.000000e+00,-9.201327e-07,0.000000e+00,-6.686726e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,4.254369,4.712440,4.836094,2.668600,3.629953,1.640128,0.4,1.7,5.200000,1.622811e+00,...,-0.000002,-5.391748e-07,-3.094541e-07,-0.000035,0.000000e+00,0.0,-3.345210e-07,-4.580856e-07,-3.435519e-07,-9.531376e-07
9996,3.210700,1.671734,4.661965,1.612852,0.490558,2.501960,0.4,1.7,3.086005,1.944606e-07,...,-0.000001,-4.541733e-07,0.000000e+00,-2.080116,-1.879487e-07,0.0,0.000000e+00,-1.067423e-06,0.000000e+00,-3.644367e-09
9997,4.979303,3.477785,4.071830,2.229018,3.314802,2.390931,0.4,1.7,5.200000,5.078266e-01,...,-0.000002,-1.077054e-06,-1.362549e-07,-0.000024,0.000000e+00,0.0,-2.538087e-07,-4.059619e-07,-1.515269e-07,-1.182981e-06
9998,2.047435,5.546288,4.167393,0.539519,4.802855,0.740298,0.4,1.7,5.200000,7.091980e-02,...,-0.000001,0.000000e+00,-8.133506e-08,-0.016308,0.000000e+00,0.0,0.000000e+00,0.000000e+00,-2.923717e-07,-4.133202e-08


In [36]:
def convert(x):
    #Remove spaces around the '+' or '-' before 'j'
    x = x.replace(" + ", "+").replace(" - ", "-").replace(" j ", "j").strip()
    return np.complex64(x)

In [37]:
""" Define input and output columns"""

load_p = list(df.columns[0:3])
load_q = list(df.columns[3:6])
inputs = load_p + load_q
print(f"Input includes {inputs}\n")

gen_p = list(df.columns[6:11])
gen_q = list(df.columns[11:16])
bus_vm = []
bus_va = []
v_bus = df.columns[21:26]
for bus_v_column in v_bus:
    df[bus_v_column + '_mag'] = df[bus_v_column].apply(convert).apply(np.abs)
    bus_vm.append(bus_v_column + '_mag')
    df[bus_v_column + '_ang'] = df[bus_v_column].apply(convert).apply(np.angle)
    df[bus_v_column + '_ang'] = -1*np.rad2deg(df[bus_v_column + '_ang'].values)
    bus_va.append(bus_v_column + '_ang')
outputs = gen_p + gen_q + bus_vm + bus_va
print(f"Output includes {outputs}\n")

Input includes ['load1:pl', 'load2:pl', 'load3:pl', 'load1:ql', 'load2:ql', 'load3:ql']

Output includes ['gen1:pg', 'gen2:pg', 'gen3:pg', 'gen4:pg', 'gen5:pg', 'gen1:qg', 'gen2:qg', 'gen3:qg', 'gen4:qg', 'gen5:qg', 'bus1:v_bus_mag', 'bus2:v_bus_mag', 'bus3:v_bus_mag', 'bus4:v_bus_mag', 'bus5:v_bus_mag', 'bus1:v_bus_ang', 'bus2:v_bus_ang', 'bus3:v_bus_ang', 'bus4:v_bus_ang', 'bus5:v_bus_ang']



In [38]:
""" Custom dataset"""

class OPFDataset(Dataset):
    def __init__(self, data, inputs, outputs):
        self.data = data
        self.inputs = inputs
        self.outputs = outputs
    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        input = torch.tensor(self.data.iloc[idx,:len(self.inputs)].values).to(torch.float32)
        output = torch.tensor(self.data.iloc[idx, len(self.inputs):].values).to(torch.float32)
        return input, output

In [39]:
data = df[inputs + outputs]
train_data = data.sample(frac=0.8, random_state=SEED)
test_data = data.drop(train_data.index)
train_data_loader = DataLoader(OPFDataset(train_data, inputs, outputs),
                              batch_size=32, shuffle=True)
test_data_loader  = DataLoader(OPFDataset(test_data, inputs, outputs),
                              batch_size=32, shuffle=False)

# Define the evaluation metric as we did in the discussion


In [40]:
from pypower.api import makeYbus, ext2int

In [41]:
class OPF_metric():
    def __init__(self, net):
        basemva = net['baseMVA']
        linear_cost = net['gencost'][:,5]
        self.cost_coef = torch.tensor(linear_cost).to(torch.float32)

        pg_max = net['gen'][:, idx_gen.PMAX] / basemva
        pg_min = net['gen'][:, idx_gen.PMIN] / basemva
        qg_max = net['gen'][:, idx_gen.QMAX] / basemva
        qg_min = net['gen'][:, idx_gen.QMIN] / basemva
        vm_max = net['bus'][:, idx_bus.VMAX]
        vm_min = net['bus'][:, idx_bus.VMIN]
        va_max = [np.pi/2 for i in range(len(net['bus']))]
        va_min = [-np.pi/2 for i in range(len(net['bus']))]
        outputs_min = np.concatenate([pg_min, qg_min, vm_min, va_min])
        outputs_max = np.concatenate([pg_max, qg_max, vm_max, va_max])
        self.outputs_min = torch.as_tensor(outputs_min).to(torch.float32).view(1,-1)
        self.outputs_max = torch.as_tensor(outputs_max).to(torch.float32).view(1,-1)

        # ============= Calculate the bus admittance matrix (Ybus) and branch admittance matrices (Yf, Yt) ============#
        net = ext2int(net)
        Ybus, Yf, Yt = makeYbus(net['baseMVA'], net['bus'], net['branch'])
        Ybus = Ybus.todense()
        self.Ybus_real = torch.as_tensor(Ybus.real).to(torch.float32)
        self.Ybus_imag = torch.as_tensor(Ybus.imag).to(torch.float32)
        Yf = Yf.todense()
        Yt = Yt.todense()
        self.Yf_real = torch.as_tensor(Yf.real).to(torch.float32)
        self.Yf_imag = torch.as_tensor(Yf.imag).to(torch.float32)
        self.Yt_real = torch.as_tensor(Yt.real).to(torch.float32)
        self.Yt_imag = torch.as_tensor(Yt.imag).to(torch.float32)

        self.gen_bus_index = net['gen'][:,idx_gen.GEN_BUS]
        self.load_bus_index = [i for i in range(5) if net['bus'][i, idx_bus.PD]>0]

        self.fbus = net['branch'][:,idx_brch.F_BUS].astype(int)
        self.tbus = net['branch'][:,idx_brch.T_BUS].astype(int)
        self.smax = torch.as_tensor(net['branch'][:,idx_brch.RATE_A] / basemva).to(torch.float32)
        self.angmax = torch.as_tensor(np.deg2rad(net['branch'][:,idx_brch.ANGMAX])).to(torch.float32)

    # Let's find the generation cost
    def cal_gen_cost(self, outputs):
        device = outputs.device  # Get the device of the outputs tensor
        cost_coef = self.cost_coef.to(device)
        pg = outputs[:, :5]
        return (pg  * cost_coef).sum(1)

    def cal_upper_lower_bound_violation(self, outputs):
        device = outputs.device  # Get the device of the outputs tensor
        outputs_max = self.outputs_max.to(device)  # Move to the same device
        outputs_min = self.outputs_min.to(device)  # Move to the same device
    
        # Use the local variables outputs_max and outputs_min
        vio_1 = torch.relu(outputs - outputs_max)
        vio_2 = torch.relu(outputs_min - outputs)
        return vio_1 + vio_2

    def gen_load_to_bus(self, inputs, outputs):
        device = outputs.device
        batch = inputs.shape[0]
        pd = inputs[:, :3]
        qd = inputs[:, 3:]
        pg = outputs[:, 0:5]
        qg = outputs[:, 5:10]
        bus_pg = torch.zeros([batch, 5], device=device)
        bus_qg = torch.zeros([batch, 5], device=device)
        for i, bus_index in enumerate(self.gen_bus_index):
            bus_pg[:, int(bus_index)] = bus_pg[:, int(bus_index)] + pg[:, i]
            bus_qg[:, int(bus_index)] = bus_qg[:, int(bus_index)] + qg[:, i]
        bus_pd = torch.zeros([batch, 5], device=device)
        bus_qd = torch.zeros([batch, 5], device=device)
        for i, bus_index in enumerate(self.load_bus_index):

            bus_pd[:, bus_index] = bus_pd[:, bus_index] + pd[:, i]
            bus_qd[:, bus_index] = bus_qd[:, bus_index] + qd[:, i]
        return bus_pg, bus_qg, bus_pd, bus_qd

    def cal_power_balance_violation(self, inputs, outputs):
        device = outputs.device 
        bus_pg, bus_qg, bus_pd, bus_qd = self.gen_load_to_bus(inputs, outputs)
        bus_p_inj = bus_pg - bus_pd
        bus_q_inj = bus_qg - bus_qd
        
        vm, va = outputs[:, 10:15], outputs[:, 15:20]
        vr = vm * torch.cos(va)
        vi = vm * torch.sin(va)
        
        self.Ybus_real = self.Ybus_real.to(device)
        self.Ybus_imag = self.Ybus_imag.to(device)

        Ir = torch.matmul(vr, self.Ybus_real) - vi @ self.Ybus_imag
        Ii = vi @ self.Ybus_real + vr @ self.Ybus_imag
        bus_p = (vr * Ir) + (vi * Ii)
        bus_q = (vi * Ir) - (vr * Ii)
        return torch.abs(bus_p_inj - bus_p), torch.abs(bus_q_inj - bus_q)

    # Let's calculate the branch flow & angle constraint violation
    def cal_branch_flow_vio(self, outputs):
        device = outputs.device 
        vm, va = outputs[:, 10:15], outputs[:, 15:20]
        vr = vm * torch.cos(va)
        vi = vm * torch.sin(va)
        
        self.Yf_real = self.Yf_real.to(device)
        self.Yf_imag = self.Yf_imag.to(device)
        self.Yt_real = self.Yt_real.to(device)
        self.Yt_imag = self.Yt_imag.to(device)
        self.smax = self.smax.to(device)
        self.angmax = self.angmax.to(device)
        
        Irf = vr @ self.Yf_real.T - vi @ self.Yf_imag.T
        Iif = vi @ self.Yf_real.T + vr @ self.Yf_imag.T
        branch_pf = vr[:, self.fbus] * Irf + vi[:, self.fbus] * Iif
        branch_qf = vi[:, self.fbus] * Irf - vr[:, self.fbus] * Iif
        sf = torch.sqrt((torch.square(branch_pf) + torch.square(branch_qf)))

        Irt = vr @ self.Yt_real.T - vi @ self.Yt_imag.T
        Iit = vi @ self.Yt_real.T + vr @ self.Yt_imag.T
        branch_pf = vr[:, self.tbus] * Irt + vi[:, self.tbus] * Iit
        branch_qf = vi[:, self.tbus] * Irt - vr[:, self.tbus] * Iit
        st = torch.sqrt((torch.square(branch_pf) + torch.square(branch_qf)))

        branch_flow = torch.maximum(sf, st)
        branch_ang = torch.abs(va[:, self.fbus] - va[:, self.tbus])
        return torch.relu(branch_flow - self.smax), torch.relu(branch_ang - self.angmax)

In [42]:
opf_metric = OPF_metric(net)

In [43]:
def test(dataloader, model, opf_metric):
    device = next(model.parameters()).device
    batch_snapshot = next(iter(dataloader))
    X, Y = batch_snapshot[0].to(device), batch_snapshot[1].to(device)
    with torch.no_grad():
      Y_pred = model(X)

    test_sol_mse = ((Y_pred - Y)**2)

    Y_pred_cost = opf_metric.cal_gen_cost(Y_pred)
    Y_opt_cost = opf_metric.cal_gen_cost(Y)
    test_obj_gap = (Y_pred_cost - Y_opt_cost)/Y_opt_cost

    vio_bound = opf_metric.cal_upper_lower_bound_violation(Y_pred)
    vio_p_mis, vio_q_mis = opf_metric.cal_power_balance_violation(X, Y_pred)
    vio_flow, vio_ang = opf_metric.cal_branch_flow_vio(Y_pred)
    eq_vio = torch.cat([vio_p_mis, vio_q_mis], dim=1)
    ineq_vio = torch.cat([vio_bound, vio_flow, vio_ang], dim=1)
    return test_sol_mse, test_obj_gap, eq_vio, ineq_vio

def model_train_test(train_dataloader, test_dataloader, model,
                     loss_fn, opf_metric, epochs = 10, penalty=False):
    torch.manual_seed(2024)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
    results = {
        "train_losses": [],
        "test_mse": [],
        "opt_gaps": [],
        "eq_vios": [],
        "ineq_vios": []
    }
    print(f"==============================================Start training on {device}===========================================")

    for t in range(epochs):
        model.train()
        train_loss = []
        with torch.enable_grad():
            for batch, (X, y) in enumerate(train_dataloader):
                X, y = X.to(device), y.to(device)
                # Compute prediction error
                pred = model(X)
                if penalty:
                  loss = loss_fn(X, pred, y)
                else:
                  loss = loss_fn(pred, y)
                # Backpropagation
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                train_loss.append(loss.item())
            train_loss = np.mean(train_loss)
            results["train_losses"].append(train_loss)
            test_sol_mse, test_obj_gap, eq_vio, ineq_vio = test(test_dataloader, model, opf_metric)
            
            results["test_mse"].append(test_sol_mse)
            results["opt_gaps"].append(test_obj_gap)
            results["eq_vios"].append(eq_vio)
            results["ineq_vios"].append(ineq_vio)
            
            print(f"Epoch {t+1} | Train loss: {train_loss:.7f} | ",
                f"Test mse: {test_sol_mse.mean():.7f} | ",
                f"opt gap: {test_obj_gap.mean():.7f}% | ",
                f"eq vio: {eq_vio.sum(1).mean():.7f} | ",
                f"ineq vio: {ineq_vio.sum(1).mean():.7f} |")
        
    return results


# ResNet

In [44]:
class residual_layer(nn.Module):
    def __init__(self, in_dim, outdim):
        super().__init__()
        """
        define a residual layer
        """
        self.layer = nn.Sequential(nn.Linear(in_dim, in_dim//2),
                                  nn.ReLU(),
                                  nn.Linear(in_dim//2,in_dim))
    def forward(self, x):
        return self.layer(x) + x


class ResidualNeuralNetwork(nn.Module):
    def __init__(self, x_size=3, y_size=5, width=32, depth=3):
        super().__init__()
        NN = [nn.Linear(x_size, width)]
        for _ in range(depth):
            NN.append(residual_layer(width, width))
        NN.append(nn.Linear(width, y_size))
        self.NN = nn.Sequential(*NN)
    def forward(self, x):
        y = self.NN(x)
        return y

# Enforcing Physical Limits

In [45]:
""" Let's add the violatio into loss function and minimize it
"""
class MSEPenaltyLoss(nn.Module):
    def __init__(self, opf_metric, w_eq=0.1, w_ineq=0.1):
        super().__init__()
        self.w_eq = w_eq
        self.w_ineq = w_ineq
        self.mse = nn.MSELoss()
        self.opf_metric = opf_metric

    def forward(self, X, pred, target):
        MSE = self.mse(pred, target)
        
        vio_bound = self.opf_metric.cal_upper_lower_bound_violation(pred)
        vio_p_mis, vio_q_mis = self.opf_metric.cal_power_balance_violation(X, pred)
        vio_flow, vio_ang = self.opf_metric.cal_branch_flow_vio(pred)

        eq_vio = torch.cat([vio_p_mis, vio_q_mis], dim=1)
        ineq_vio = torch.cat([vio_bound, vio_flow, vio_ang], dim=1)

        loss = MSE + self.w_eq * eq_vio.mean() + self.w_ineq * ineq_vio.mean()
        return loss

In [46]:
""" AC-OPF solutions must respect operational constraints
        - voltage levels
        - power flow litmits
    which are essential for maintaining the physical and safety standards of power systems.
    We will try to implement different approaches to enfore the physical limits of AC-OPF
        1. Upper/Lower Bound Constraint: 
          - Use the sigmoid activation function and scaling operations to enforce simple upper and lower bounds.
        2. Non-linear Equality and Inequality Constraints:
          - Improve feasibility by intergrating penalty functions into the loss function.
          - Adjust the penalty weights to enhance model performance and constraint adherence
"""

' AC-OPF solutions must respect operational constraints\n        - voltage levels\n        - power flow litmits\n    which are essential for maintaining the physical and safety standards of power systems.\n    We will try to implement different approaches to enfore the physical limits of AC-OPF\n        1. Upper/Lower Bound Constraint: \n          - Use the sigmoid activation function and scaling operations to enforce simple upper and lower bounds.\n        2. Non-linear Equality and Inequality Constraints:\n          - Improve feasibility by intergrating penalty functions into the loss function.\n          - Adjust the penalty weights to enhance model performance and constraint adherence\n'

In [47]:
""" Suppose we have the upper/lower bound information for the decision varibale.
    Let's try to get the NN to enforce the bound. This involves two steps:
        1. Using sigmoid function on the output, to make it within [0, 1]
        2. Using affine transformation to make the [0, 1] output to the corresponding [I,u]
"""

" Suppose we have the upper/lower bound information for the decision varibale.\n    Let's try to get the NN to enforce the bound. This involves two steps:\n        1. Using sigmoid function on the output, to make it within [0, 1]\n        2. Using affine transformation to make the [0, 1] output to the corresponding [I,u]\n"

# ResNet

In [48]:
class BoundNeuralNetwork(ResidualNeuralNetwork):
    def __init__(self, x_size, y_size, width, depth, yl, yu):
        super().__init__(x_size, y_size, width, depth)
 
                # Store bounds as buffers for consistency across devices
        self.register_buffer('yl', torch.tensor(yl, dtype=torch.float32))
        self.register_buffer('yu', torch.tensor(yu, dtype=torch.float32))
    def forward(self, x):
        y = self.NN(x)
        y = torch.sigmoid(y)
        y = y * (self.yu - self.yl) + self.yl
        return y

In [49]:
model_bound = BoundNeuralNetwork(x_size=len(inputs),
                                y_size=len(outputs),
                                width=64,
                                depth=6,
                                yl=opf_metric.outputs_min,
                                yu=opf_metric.outputs_max,
                                ).to(device)

/tmp/ipykernel_15893/4289910525.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer('yl', torch.tensor(yl, dtype=torch.float32))
/tmp/ipykernel_15893/4289910525.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer('yu', torch.tensor(yu, dtype=torch.float32))


In [50]:
result2 = model_train_test(train_data_loader, test_data_loader, model_bound, MSEPenaltyLoss(opf_metric),opf_metric, epochs=50, penalty=True)

==============================================Start training on cuda===========================================
Epoch 1 | Train loss: 1.0253067 |  Test mse: 0.1729798 |  opt gap: 0.0265084% |  eq vio: 24.6351624 |  ineq vio: 0.6883281 |
Epoch 2 | Train loss: 0.3101340 |  Test mse: 0.1527106 |  opt gap: 0.0256341% |  eq vio: 13.0952320 |  ineq vio: 0.0725400 |
Epoch 3 | Train loss: 0.2630629 |  Test mse: 0.1460355 |  opt gap: 0.0260687% |  eq vio: 8.7753563 |  ineq vio: 0.3406026 |
Epoch 4 | Train loss: 0.2414360 |  Test mse: 0.1345419 |  opt gap: 0.0181786% |  eq vio: 12.1189709 |  ineq vio: 0.8485147 |
Epoch 5 | Train loss: 0.2198617 |  Test mse: 0.1252365 |  opt gap: 0.0087278% |  eq vio: 6.2302494 |  ineq vio: 0.4768497 |
Epoch 6 | Train loss: 0.2078988 |  Test mse: 0.1116590 |  opt gap: 0.0093311% |  eq vio: 14.6014652 |  ineq vio: 3.0181298 |
Epoch 7 | Train loss: 0.1881244 |  Test mse: 0.0975623 |  opt gap: 0.0065881% |  eq vio: 6.7103801 |  ineq vio: 0.3134676 |
Epoch 8 | Train 

In [51]:
model_res = ResidualNeuralNetwork(x_size=len(inputs),
                                  y_size=len(outputs),
                                  width=64,
                                  depth=6).to(device)

a = model_train_test(train_data_loader, test_data_loader, model_res, nn.MSELoss(), opf_metric, epochs=50)

==============================================Start training on cuda===========================================
Epoch 1 | Train loss: 0.3726798 |  Test mse: 0.1459895 |  opt gap: 0.0123390% |  eq vio: 105.0272446 |  ineq vio: 32.4881058 |
Epoch 2 | Train loss: 0.1032014 |  Test mse: 0.0767267 |  opt gap: 0.0155402% |  eq vio: 113.4449005 |  ineq vio: 35.0181732 |
Epoch 3 | Train loss: 0.0601017 |  Test mse: 0.0673808 |  opt gap: -0.0088588% |  eq vio: 72.6319275 |  ineq vio: 16.2297115 |
Epoch 4 | Train loss: 0.0502787 |  Test mse: 0.0557695 |  opt gap: -0.0143666% |  eq vio: 76.9824677 |  ineq vio: 22.3549652 |
Epoch 5 | Train loss: 0.0414508 |  Test mse: 0.0533770 |  opt gap: 0.0030453% |  eq vio: 66.8304367 |  ineq vio: 17.8328362 |
Epoch 6 | Train loss: 0.0352313 |  Test mse: 0.0555628 |  opt gap: -0.0270800% |  eq vio: 63.6776505 |  ineq vio: 12.9448967 |
Epoch 7 | Train loss: 0.0319136 |  Test mse: 0.0469129 |  opt gap: -0.0036366% |  eq vio: 48.3069954 |  ineq vio: 8.7038841 |
E

{'train_losses': [0.37267980828881264,
  0.10320139509439469,
  0.060101729184389115,
  0.05027872336655855,
  0.04145084691792727,
  0.035231254808604714,
  0.03191361517086625,
  0.02800167289376259,
  0.02669259575009346,
  0.02532236912474036,
  0.02390420011803508,
  0.024137804517522454,
  0.023366205386817454,
  0.022029305892065167,
  0.02129560793377459,
  0.020240311598405242,
  0.019445132953114806,
  0.020397845385596157,
  0.01847060682810843,
  0.017819520263932645,
  0.018093730382621288,
  0.016922882163897156,
  0.016762671461328864,
  0.016148173727095125,
  0.01620698005799204,
  0.016247864903882147,
  0.015771064506843686,
  0.015402019293047488,
  0.01378941469360143,
  0.014699292388744652,
  0.014490492088720202,
  0.013841680405661463,
  0.013435832071118058,
  0.014029553822241723,
  0.014515363225713371,
  0.012950294625945389,
  0.014401263574138283,
  0.012030673330649734,
  0.011241766932420433,
  0.013602074710652232,
  0.012114936096593738,
  0.011204910

# Model CNN

In [52]:
class ConvolutionNetwork(nn.Module):
    def __init__(self, x_size, y_size, num_filter=32, kernel_size=3, depth=3):
        super().__init__()
        self.conv_layers = nn.ModuleList()

        self.conv_layers.append(nn.Conv1d(in_channels=1, out_channels=num_filter,kernel_size=kernel_size, padding=1))
        for _ in range(depth):
            self.conv_layers.append(nn.Conv1d(in_channels=num_filter, out_channels=num_filter, kernel_size=kernel_size, padding=1))

        self.fc1 = nn.Linear(num_filter * x_size, 128)
        self.fc2 = nn.Linear(128, y_size)

    def forward(self, x):
        x = x.unsqueeze(1)
        for conv in self.conv_layers:
            x = torch.nn.functional.relu(conv(x))

        x = x.view(x.size(0), -1)
        x = torch.nn.functional.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [53]:
model_cnn = ConvolutionNetwork(x_size=len(inputs),
                              y_size=len(outputs),
                              num_filter=64,
                              kernel_size=3,
                              depth=2).to(device)


In [63]:
results_2 = model_train_test(train_data_loader, test_data_loader, model_cnn, MSEPenaltyLoss(opf_metric),opf_metric, epochs=100, penalty=True)

==============================================Start training on cuda===========================================
Epoch 1 | Train loss: 0.0469010 |  Test mse: 0.0260672 |  opt gap: -0.0134344% |  eq vio: 2.3747652 |  ineq vio: 0.1468426 |
Epoch 2 | Train loss: 0.0340239 |  Test mse: 0.0205036 |  opt gap: -0.0089318% |  eq vio: 2.8188467 |  ineq vio: 0.0532132 |
Epoch 3 | Train loss: 0.0353772 |  Test mse: 0.0194423 |  opt gap: 0.0145895% |  eq vio: 2.5199761 |  ineq vio: 0.0910564 |
Epoch 4 | Train loss: 0.0295974 |  Test mse: 0.0235440 |  opt gap: -0.0028397% |  eq vio: 3.7911451 |  ineq vio: 0.1169322 |
Epoch 5 | Train loss: 0.0296833 |  Test mse: 0.0274494 |  opt gap: -0.0233585% |  eq vio: 2.4001877 |  ineq vio: 0.0990151 |
Epoch 6 | Train loss: 0.0296253 |  Test mse: 0.0167946 |  opt gap: 0.0045387% |  eq vio: 1.9588389 |  ineq vio: 0.1640822 |
Epoch 7 | Train loss: 0.0285837 |  Test mse: 0.0229337 |  opt gap: -0.0182824% |  eq vio: 2.2597280 |  ineq vio: 0.0806585 |
Epoch 8 | Train

KeyboardInterrupt: 

In [55]:
results_3 = model_train_test(train_data_loader, test_data_loader, model_cnn, nn.MSELoss(),opf_metric, epochs=50, penalty=False)

==============================================Start training on cuda===========================================
Epoch 1 | Train loss: 0.0175382 |  Test mse: 0.0387685 |  opt gap: -0.0299839% |  eq vio: 3.2078347 |  ineq vio: 0.4683458 |
Epoch 2 | Train loss: 0.0150754 |  Test mse: 0.0338906 |  opt gap: -0.0107389% |  eq vio: 2.9027910 |  ineq vio: 0.2604286 |
Epoch 3 | Train loss: 0.0153919 |  Test mse: 0.0371200 |  opt gap: 0.0075507% |  eq vio: 3.3280730 |  ineq vio: 0.2333445 |
Epoch 4 | Train loss: 0.0128469 |  Test mse: 0.0247826 |  opt gap: 0.0205374% |  eq vio: 4.3697271 |  ineq vio: 0.1310766 |
Epoch 5 | Train loss: 0.0129395 |  Test mse: 0.0338513 |  opt gap: -0.0210325% |  eq vio: 3.6292424 |  ineq vio: 0.3260539 |
Epoch 6 | Train loss: 0.0134695 |  Test mse: 0.0272655 |  opt gap: 0.0168451% |  eq vio: 4.5053144 |  ineq vio: 0.1942282 |
Epoch 7 | Train loss: 0.0130876 |  Test mse: 0.0351490 |  opt gap: -0.0221618% |  eq vio: 5.2860394 |  ineq vio: 0.1906422 |
Epoch 8 | Train 

# Model RNN

In [56]:
class RecurrentNetwork(nn.Module):
    def __init__(self, x_size, y_size, hidden_dim, num_layers):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # LSTM layer
        self.rnn = nn.LSTM(input_size=x_size, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, y_size)

    def forward(self, x):
        # x shape: (batch_size, seq_size, x_size)
        if x.dim() == 2:
            x = x.unsqueeze(1)
        # Initialize hidden state and cell state
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)

        out, _ = self.rnn(x, (h0, c0)) # outshape: (batch_size, seq_len, hidden_dim)
        out = out[:, -1, :] #(batch_size, hidden_dim)
        out = self.fc(out)
        return out
    

In [57]:
model_rnn = RecurrentNetwork(x_size=len(inputs),
                             y_size=len(outputs),
                             hidden_dim=32,
                             num_layers=2).to(device)

In [58]:
resutl_4 = model_train_test(train_data_loader, test_data_loader, model_rnn, nn.MSELoss(), opf_metric, epochs=50)

==============================================Start training on cuda===========================================
Epoch 1 | Train loss: 1.6954482 |  Test mse: 0.3561234 |  opt gap: -0.0279395% |  eq vio: 11.7757988 |  ineq vio: 0.0490910 |
Epoch 2 | Train loss: 0.2886111 |  Test mse: 0.3282022 |  opt gap: 0.0309470% |  eq vio: 9.3524494 |  ineq vio: 0.0543624 |
Epoch 3 | Train loss: 0.2810066 |  Test mse: 0.3219217 |  opt gap: 0.0273309% |  eq vio: 10.4592381 |  ineq vio: 0.0575419 |
Epoch 4 | Train loss: 0.2718753 |  Test mse: 0.3119617 |  opt gap: 0.0184183% |  eq vio: 13.4044819 |  ineq vio: 0.2383816 |
Epoch 5 | Train loss: 0.2553548 |  Test mse: 0.2800147 |  opt gap: 0.0088912% |  eq vio: 25.8451080 |  ineq vio: 3.1909409 |
Epoch 6 | Train loss: 0.2065717 |  Test mse: 0.2178358 |  opt gap: 0.0138423% |  eq vio: 25.4468575 |  ineq vio: 2.2864606 |
Epoch 7 | Train loss: 0.1630523 |  Test mse: 0.1711095 |  opt gap: 0.0023964% |  eq vio: 28.7024403 |  ineq vio: 3.8419907 |
Epoch 8 | Tra

In [59]:
resutl_5 = model_train_test(train_data_loader, test_data_loader, model_rnn, MSEPenaltyLoss(opf_metric), opf_metric, epochs=50, penalty=True)

==============================================Start training on cuda===========================================
Epoch 1 | Train loss: 0.0756021 |  Test mse: 0.0401667 |  opt gap: 0.0089543% |  eq vio: 4.4033384 |  ineq vio: 0.1371771 |
Epoch 2 | Train loss: 0.0587954 |  Test mse: 0.0403263 |  opt gap: 0.0015840% |  eq vio: 2.8320074 |  ineq vio: 0.1289976 |
Epoch 3 | Train loss: 0.0539948 |  Test mse: 0.0424130 |  opt gap: -0.0073716% |  eq vio: 2.1543694 |  ineq vio: 0.1214719 |
Epoch 4 | Train loss: 0.0499817 |  Test mse: 0.0413913 |  opt gap: 0.0015000% |  eq vio: 2.2238343 |  ineq vio: 0.1501550 |
Epoch 5 | Train loss: 0.0469162 |  Test mse: 0.0409530 |  opt gap: -0.0045310% |  eq vio: 1.9141877 |  ineq vio: 0.1512432 |
Epoch 6 | Train loss: 0.0463906 |  Test mse: 0.0404935 |  opt gap: -0.0017007% |  eq vio: 1.6132426 |  ineq vio: 0.1338589 |
Epoch 7 | Train loss: 0.0460059 |  Test mse: 0.0402311 |  opt gap: 0.0016525% |  eq vio: 2.7447681 |  ineq vio: 0.1395830 |
Epoch 8 | Train l

# MLP model

In [60]:
class NeuralNetwork(nn.Module):
    def __init__(self, x_size=3, y_size=5, width=32, depth=3):
        super().__init__()
        NN = [nn.Linear(x_size, width)]
        for _ in range(depth):
            NN.append(nn.Linear(width, width))
            NN.append(nn.ReLU())
        NN.append(nn.Linear(width, y_size))
        self.NN = nn.Sequential(*NN)
    def forward(self, x):
        y = self.NN(x)
        return y

In [61]:
model_base = NeuralNetwork(x_size=len(inputs),
                          y_size=len(outputs),
                          width=32,
                          depth=2).to(device)

results = model_train_test(train_data_loader, test_data_loader, model_base, nn.MSELoss(), opf_metric, epochs=50)

==============================================Start training on cuda===========================================
Epoch 1 | Train loss: 0.8852510 |  Test mse: 0.2678800 |  opt gap: 0.0211753% |  eq vio: 225.4129333 |  ineq vio: 88.8420563 |
Epoch 2 | Train loss: 0.1874891 |  Test mse: 0.1689313 |  opt gap: 0.0086749% |  eq vio: 123.3648376 |  ineq vio: 43.5044479 |
Epoch 3 | Train loss: 0.1561301 |  Test mse: 0.1517129 |  opt gap: 0.0237959% |  eq vio: 81.9214249 |  ineq vio: 21.7540398 |
Epoch 4 | Train loss: 0.1425775 |  Test mse: 0.1455908 |  opt gap: 0.0149567% |  eq vio: 71.0983963 |  ineq vio: 18.6453400 |
Epoch 5 | Train loss: 0.1298699 |  Test mse: 0.1402443 |  opt gap: 0.0186115% |  eq vio: 66.2074280 |  ineq vio: 15.7131662 |
Epoch 6 | Train loss: 0.1150909 |  Test mse: 0.1175856 |  opt gap: 0.0233613% |  eq vio: 59.1141396 |  ineq vio: 13.5611248 |
Epoch 7 | Train loss: 0.0965847 |  Test mse: 0.1025167 |  opt gap: 0.0189477% |  eq vio: 62.3640060 |  ineq vio: 14.7731209 |
Epoc

In [62]:
results_1 = model_train_test(train_data_loader, test_data_loader, model_base, MSEPenaltyLoss(opf_metric), opf_metric, epochs=50, penalty=True)

==============================================Start training on cuda===========================================
Epoch 1 | Train loss: 0.1478204 |  Test mse: 0.0447742 |  opt gap: -0.0003002% |  eq vio: 6.3838167 |  ineq vio: 0.3077147 |
Epoch 2 | Train loss: 0.1129617 |  Test mse: 0.0471497 |  opt gap: -0.0010167% |  eq vio: 6.7282553 |  ineq vio: 0.2128766 |
Epoch 3 | Train loss: 0.1074474 |  Test mse: 0.0477372 |  opt gap: -0.0028505% |  eq vio: 5.5179272 |  ineq vio: 0.3712055 |
Epoch 4 | Train loss: 0.1014206 |  Test mse: 0.0471158 |  opt gap: -0.0019473% |  eq vio: 5.6106439 |  ineq vio: 0.2431101 |
Epoch 5 | Train loss: 0.1050206 |  Test mse: 0.0462440 |  opt gap: -0.0013022% |  eq vio: 4.1881967 |  ineq vio: 0.2683710 |
Epoch 6 | Train loss: 0.0973152 |  Test mse: 0.0444487 |  opt gap: 0.0024465% |  eq vio: 4.0629616 |  ineq vio: 0.1386177 |
Epoch 7 | Train loss: 0.0924564 |  Test mse: 0.0444705 |  opt gap: -0.0008569% |  eq vio: 3.8357878 |  ineq vio: 0.3323562 |
Epoch 8 | Trai